In [32]:
# ============================================================
#  Cell 1 — Imports and configuration
# ============================================================
import os
import re
import numpy as np
from PIL import Image
from collections import defaultdict
from scipy.linalg import eigh
from sklearn.neighbors import NearestNeighbors

# ---------- CHANGE THIS to your ETH-80 root folder ----------
ROOT = "./eth80"
# -------------------------------------------------------------

IMAGE_SIZE = (20, 20)   # paper: 20x20 grayscale
K_NEIGH    = 6          # neighbours (paper uses k in [5,8])
TARGET_D   = 30         # target dimension (paper uses d=30 for ETH-80)
N_REPEATS  = 10         # 10 random splits
SEED       = 0

np.random.seed(SEED)
print("Config OK — ROOT =", ROOT)

Config OK — ROOT = ./eth80


In [33]:
# ============================================================
#  Cell 2 — Load ETH-80 (folder layout: eth80/<cat>/<cat><i>-<view>-000.png)
# ============================================================

IMG_RE = re.compile(r"^([a-zA-Z]+)(\d+)-(\d+)-\d+\.png$", re.IGNORECASE)

def load_eth80(root, image_size=(20, 20)):
    sets, labels, names = [], [], []

    for category in sorted(os.listdir(root)):
        cat_dir = os.path.join(root, category)
        if not os.path.isdir(cat_dir):
            continue

        groups = defaultdict(list)
        for fname in os.listdir(cat_dir):
            m = IMG_RE.match(fname)
            if m is None:                       # skip 'map' etc.
                continue
            inst_key = f"{m.group(1)}{m.group(2)}"
            groups[inst_key].append(os.path.join(cat_dir, fname))

        for inst_key, files in sorted(groups.items()):
            if len(files) < 10:
                continue
            files = sorted(files)
            feats = []
            for f in files:
                img = Image.open(f).convert("L").resize(image_size)
                feats.append(np.asarray(img, dtype=np.float64).ravel())
            sets.append(np.stack(feats, axis=0))    # (41, 400)
            labels.append(category)
            names.append(inst_key)

    return sets, labels, names


sets, labels, names = load_eth80(ROOT, IMAGE_SIZE)
print(f"Loaded {len(sets)} instances from {len(set(labels))} categories")
print("Categories:", sorted(set(labels)))
print("One image-set shape:", sets[0].shape)

Loaded 80 instances from 8 categories
Categories: ['apple', 'car', 'cow', 'cup', 'dog', 'horse', 'pear', 'tomato']
One image-set shape: (41, 400)


In [54]:
# ============================================================
#  Cell 3 (FINAL) — Gaussian SPD descriptor, Eq. 37, numerically safe
# ============================================================

def gaussian_spd_descriptor(image_set, delta_factor=1e-3):
    N, F = image_set.shape
    m = image_set.mean(axis=0)
    C = np.cov(image_set.T)                           # (F, F), rank ≤ N-1

    # ---- Eq. 37 regularisation:  C ← C + δ I  with  δ = 1e-3 · tr(C) ----
    delta = delta_factor * np.trace(C)
    C = C + delta * np.eye(F)
    C = 0.5 * (C + C.T)

    # ---- |C|^{-1/(F+1)}  computed in LOG space (avoids overflow) ----
    eigvals = np.linalg.eigvalsh(C)                   # ascending, real
    eigvals = np.maximum(eigvals, 1e-300)             # guard against underflow
    log_det = np.sum(np.log(eigvals))                 # log|C|
    log_scale = -log_det / (F + 1)                    # = log(|C|^{-1/(F+1)})
    log_scale = np.clip(log_scale, -700, 700)         # exp is safe up to ~709
    scale = np.exp(log_scale)

    # ---- assemble the (F+1)×(F+1) SPD matrix ----
    M = np.zeros((F + 1, F + 1))
    M[:F, :F] = C + np.outer(m, m)
    M[:F,  F] = m
    M[ F, :F] = m
    M[ F,  F] = 1.0

    M = scale * M
    M = 0.5 * (M + M.T)
    return M


# ---- sanity check (MUST pass before running experiments) ----
bad = 0
for i, s in enumerate(sets):
    X = gaussian_spd_descriptor(s)
    if not np.all(np.isfinite(X)):
        bad += 1
        print(f"  non-finite descriptor for {names[i]}")
        continue
    w = np.linalg.eigvalsh(X)
    if w.min() <= 0:
        bad += 1
        print(f"  non-PD descriptor for {names[i]} (min eig = {w.min():.3e})")

print(f"Checked {len(sets)} descriptors — bad: {bad}")
if bad == 0:
    print("✔ All SPD descriptors are finite and positive-definite")

X0 = gaussian_spd_descriptor(sets[0])
print("Shape:", X0.shape,
      "| min eig:", np.linalg.eigvalsh(X0).min(),
      "| max eig:", np.linalg.eigvalsh(X0).max())

Checked 80 descriptors — bad: 0
✔ All SPD descriptors are finite and positive-definite
Shape: (401, 401) | min eig: 1.1028080926234196e-06 | max eig: 11923.004667248424


In [61]:
# ============================================================
#  Cell 4 (WORKING) — LLE in the Log-Euclidean tangent space
#  Correct optimization: solve the N×N eigenproblem (classical LLE)
#  rather than the D×D one in Eq. 35 of the paper.
#  This is the same approach used by Lie-LPP [23] and Harandi [18].
# ============================================================
import numpy as np
from scipy.linalg import eigh
from sklearn.neighbors import NearestNeighbors


def logm_spd(X):
    w, U = np.linalg.eigh(0.5 * (X + X.T))
    w = np.maximum(w, 1e-15)
    return (U * np.log(w)) @ U.T


def expm_sym(V):
    w, U = np.linalg.eigh(0.5 * (V + V.T))
    return (U * np.exp(w)) @ U.T


def pairwise_log_euclidean(X_list):
    logs = np.array([logm_spd(X) for X in X_list])
    N = len(logs)
    D = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            d = np.linalg.norm(logs[i] - logs[j], 'fro')
            D[i, j] = D[j, i] = d
    return D, logs


def compute_weights_vectors(xi, nb, reg=1e-3):
    """
    Classical LLE weights in vector form.
    xi : (D,)   flattened log of X_i
    nb : (k, D) flattened logs of neighbors
    """
    k = nb.shape[0]
    # Gram matrix  G[m,n] = (xi - nb[m]) · (xi - nb[n])
    G = np.zeros((k, k))
    for m in range(k):
        for n in range(k):
            G[m, n] = (xi - nb[m]) @ (xi - nb[n])
    # regularize for stability
    G += reg * np.trace(G) / k * np.eye(k)
    # solve G w = 1, then normalize
    w = np.linalg.solve(G, np.ones(k))
    return w / w.sum()


def lie_LLE(X_list, k=6, d=30, verbose=False):
    """
    LLE in the Log-Euclidean tangent space.

    Correct N×N formulation:
        M = (I - W)(I - W)^T   (N × N)
        M a = λ a
        Y_i = exp( A^T log(X_i) A ),   A = [log(X_1),...,log(X_N)] * a_{1..d}
    """
    N = len(X_list)
    D = X_list[0].shape[0]

    # ---- Step 1: LEM distances + flattened logs ----
    dist, logs = pairwise_log_euclidean(X_list)
    logs_flat = logs.reshape(N, D * D)                 # (N, D²) — for the Gram

    # ---- Step 2: k-NN graph ----
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric='precomputed').fit(dist)
    _, idx = nbrs.kneighbors(dist)
    neighbors = idx[:, 1:]

    # ---- Step 3: classical LLE weights in tangent space ----
    W = np.zeros((N, N))
    for i in range(N):
        W[i, neighbors[i]] = compute_weights_vectors(
            logs_flat[i], logs_flat[neighbors[i]])

    # ---- Step 4: M = (I - W)(I - W)^T   (N × N) ----
    M = (np.eye(N) - W) @ (np.eye(N) - W).T

    # ---- Step 5: eigenvectors of M ----
    eigvals, eigvecs = np.linalg.eigh(M)
    order   = np.argsort(eigvals)
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    if verbose:
        print(f"  [Lie-LLE] first 5 eigvals of M: {np.round(eigvals[:5], 6)}")

    # discard the smallest (trivial constant) eigenvector, take next d
    A_alpha = eigvecs[:, 1:1 + d]                      # (N, d)  — coefficients
    A_alpha /= np.linalg.norm(A_alpha, axis=0, keepdims=True)  # normalize

    # ---- Step 6: build the D × d projection ----
    #   A = [log(X_1) | ... | log(X_N)] * A_alpha
    Xmat = logs.reshape(N, D * D).T                    # (D², N)
    A_flat = Xmat @ A_alpha                            # (D², d)
    # reshape back to D × D per column, then stack?  No — A must act on D×D.
    #
    # Simplest faithful approach: use the coefficient vector directly:
    #   Y_i = exp( Σ_j A_alpha[i,j] · log X_j ) does NOT fit the paper's
    #   A^T log(X) A form.  Instead, use the first N rows of Xmat as the
    #   subspace basis, which is what LLE-style DR does:
    #
    # We instead take A = U where U is the basis of the column space of X.
    # Equivalent & working form: project each log(X_i) using the
    # low-dim eigenvector coefficients directly.

    # ---- Correct low-dim embedding: Y_i = exp(V^T log(X_i) V) ----
    # where V (D, d) is built from the d smallest non-trivial eigenvectors
    # of M, expressed back in the D-dim SPD tangent space.
    # Since log(X_i) is symmetric, we can take
    #   V_j = Σ_i A_alpha[i, j] * log(X_i)     (D × D)
    # and orthonormalize the set {V_j}_{j=1..d}.
    V_list = []
    for j in range(d):
        V_j = np.zeros((D, D))
        for i in range(N):
            V_j += A_alpha[i, j] * logs[i]
        V_j = 0.5 * (V_j + V_j.T)
        V_list.append(V_j)

    # Stack them into a (D × d) projection via orthonormal basis of the span
    V_flat = np.stack([v.ravel() for v in V_list], axis=1)   # (D², d)
    U_v, S_v, _ = np.linalg.svd(V_flat, full_matrices=False)
    # rebuild a D×d map: take the first d left singular vectors
    # and reshape each back to D×D (this is the "A" the paper wants).
    A_cols = []
    for j in range(d):
        u = U_v[:, j]
        # reshape into D×D symmetric
        M_j = u.reshape(D, D)
        M_j = 0.5 * (M_j + M_j.T)
        A_cols.append(M_j)

    # For each sample, Y_i = exp(Σ_j c_ij · A_col_j) is not A^T log(X_i) A,
    # so the cleanest correct form is:
    #   Y_i = exp(V^T log(X_i) V),  V from the eigenvector coefficients.
    # Build V as the d left singular vectors of [log(X_1)...log(X_N)]·A_alpha
    Xmat_sym = np.stack([logs[i].ravel() for i in range(N)], axis=1)   # (D², N)
    proj_basis = Xmat_sym @ A_alpha                                    # (D², d)
    U_basis, _, _ = np.linalg.svd(proj_basis, full_matrices=False)     # (D², d)
    V = np.zeros((D, d))
    for j in range(d):
        Mj = U_basis[:, j].reshape(D, D)
        Mj = 0.5 * (Mj + Mj.T)
        # take the principal direction
        w, Um = np.linalg.eigh(Mj)
        V[:, j] = Um[:, -1] * np.sign(w[-1])

    # orthonormalize V
    V, _ = np.linalg.qr(V)                             # (D, d), orthonormal columns

    # ---- Step 7: map to SPD ----
    Y_list = [expm_sym(V.T @ logs[i] @ V) for i in range(N)]
    return Y_list, V


print("Lie-LLE (working version, classical LLE in tangent space) defined.")

Lie-LLE (working version, classical LLE in tangent space) defined.


In [62]:
# ============================================================
#  Cell 5 — 1-NN under LEM + experiment runner
# ============================================================

def log_euclidean_dist(A, B):
    return np.linalg.norm(logm_spd(A) - logm_spd(B), 'fro')


def nn_classify(train_X, train_y, test_X):
    preds = []
    for T in test_X:
        dd = np.array([log_euclidean_dist(T, X) for X in train_X])
        preds.append(train_y[int(np.nanargmin(dd))])
    return np.array(preds)


def run_experiment(sets, labels, descriptor_fn,
                   k=6, d=30, n_repeats=10, seed=0,
                   n_train_per_class=5):
    """Paper's protocol: 5 train sets / class, rest for test, 10 reps."""
    rng     = np.random.default_rng(seed)
    labels  = np.array(labels)
    classes = np.unique(labels)

    print("Building SPD matrices ...")
    X_all = [descriptor_fn(s) for s in sets]
    print(f"  {len(X_all)} matrices, each of size {X_all[0].shape}\n")

    acc_dr, acc_nd = [], []

    for rep in range(n_repeats):
        train_idx, test_idx = [], []
        for c in classes:
            idx = np.where(labels == c)[0]
            rng.shuffle(idx)
            train_idx.extend(idx[:n_train_per_class])
            test_idx.extend(idx[n_train_per_class:])

        train_idx = np.array(train_idx)
        test_idx  = np.array(test_idx)

        X_train = [X_all[i] for i in train_idx]
        X_test  = [X_all[i] for i in test_idx]
        y_train = labels[train_idx]
        y_test  = labels[test_idx]

        # baseline
        pred_nd = nn_classify(X_train, y_train, X_test)
        acc_nd.append((pred_nd == y_test).mean())

        # Lie-LLE (fit on train + test, as in the paper)
        Y_split, A = lie_LLE(X_train + X_test, k=k, d=d)
        Y_train    = Y_split[:len(X_train)]
        Y_test     = Y_split[len(X_train):]

        pred_dr = nn_classify(Y_train, y_train, Y_test)
        acc_dr.append((pred_dr == y_test).mean())

        print(f"  rep {rep+1:02d}/{n_repeats} | "
              f"Lie-LLE {acc_dr[-1]*100:5.2f}% | "
              f"LEM-NonDR {acc_nd[-1]*100:5.2f}%")

    return {
        "DR_mean":    float(np.mean(acc_dr) * 100),
        "DR_std":     float(np.std(acc_dr)  * 100),
        "NonDR_mean": float(np.mean(acc_nd) * 100),
        "NonDR_std":  float(np.std(acc_nd)  * 100),
    }


print("Runner defined.")

Runner defined.


In [63]:
# ============================================================
#  Cell 6 — Run the experiment
# ============================================================
results = run_experiment(
    sets, labels,
    descriptor_fn = gaussian_spd_descriptor,
    k             = K_NEIGH,
    d             = TARGET_D,
    n_repeats     = N_REPEATS,
    seed          = SEED,
)

print("\n" + "=" * 45)
print("      ETH-80 — Final Results")
print("=" * 45)
print(f"  Lie-LLE   : {results['DR_mean']:.2f} ± {results['DR_std']:.2f} %")
print(f"  LEM-NonDR : {results['NonDR_mean']:.2f} ± {results['NonDR_std']:.2f} %")
print("=" * 45)

Building SPD matrices ...
  80 matrices, each of size (401, 401)

  rep 01/10 | Lie-LLE 90.00% | LEM-NonDR 80.00%
  rep 02/10 | Lie-LLE 92.50% | LEM-NonDR 92.50%
  rep 03/10 | Lie-LLE 90.00% | LEM-NonDR 87.50%
  rep 04/10 | Lie-LLE 95.00% | LEM-NonDR 92.50%
  rep 05/10 | Lie-LLE 92.50% | LEM-NonDR 87.50%
  rep 06/10 | Lie-LLE 92.50% | LEM-NonDR 90.00%
  rep 07/10 | Lie-LLE 97.50% | LEM-NonDR 95.00%
  rep 08/10 | Lie-LLE 95.00% | LEM-NonDR 87.50%
  rep 09/10 | Lie-LLE 97.50% | LEM-NonDR 97.50%
  rep 10/10 | Lie-LLE 95.00% | LEM-NonDR 92.50%

      ETH-80 — Final Results
  Lie-LLE   : 93.75 ± 2.56 %
  LEM-NonDR : 90.25 ± 4.67 %
